In [1]:
import sys, os
import matplotlib.pyplot as plt
import seaborn as sns
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:/", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")
sns.set_style("darkgrid")

Added to sys.path:/ /Users/nirvana/Documents/NYUQuantClub1
Fixed Income Library is loaded.


### Create Build Method Collection

In [2]:
bm_list = []
# create yield curve build method
content_sofr = {
    'TARGET' : 'SOFR-1B',
    'OVERNIGHT INDEX SWAP' : 'USD-SOFR-OIS'}
content_sofr_funding = {
    'TARGET' : 'SOFR-1B-FLAT',
    'SPREAD ZERO RATE' : 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD'}
content_eonia = {
    'TARGET' : 'EONIA-1B',
    'OVERNIGHT INDEX SWAP' : 'EUR-OIS'}
content_eonia_funding = {
    'TARGET' : 'EONIA-1B-FLAT',
    'SPREAD ZERO RATE' : 'EONIA-1B-FLAT-OVER-EONIA-1B-ZERO-SPREAD'}
content_eur_usd = {
    'TARGET' : 'EUR-USD',
    'FX SPOT RATE' : 'EUR-USD'
}
# create yield curve common build method 
content_usd = {
    'TARGET' : 'USD',
    'FUNDING PARAMETERS' : 'USD-FUNDING-PARAMETERS',
    'SOLVER' : 'BRENTQ'}
# create yield curve common build method 
content_eur = {
    'TARGET' : 'EUR',
    'FUNDING PARAMETERS' : 'EUR-FUNDING-PARAMETERS',
    'SOLVER' : 'BRENTQ'}
# pack up
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_INDEX', content_sofr))
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_FUNDING', content_sofr_funding))
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_INDEX', content_eonia))
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_FUNDING', content_eonia_funding))
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_FX', content_eur_usd))
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_COMMON', content_usd))
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_COMMON', content_eur))
build_method_collection = qfCreateModelBuildMethodCollection(bm_list)

### Create Data Collection

In [3]:
### ois swap (usd)
df_swap_sofr = pd.DataFrame([    
    ['4Y', 0.03358],
    ['5Y', 0.03422]],
    columns=['axis1', 'values']).set_index('axis1')
data_swap_sofr = qfCreateData1D('OVERNIGHT INDEX SWAP', 'USD-SOFR-OIS', df_swap_sofr)
### ois swap (eur)
df_swap_eonia = pd.DataFrame([    
    ['4Y', 0.02358],
    ['5Y', 0.03422]],
    columns=['axis1', 'values']).set_index('axis1')
data_swap_eonia = qfCreateData1D('OVERNIGHT INDEX SWAP', 'EUR-OIS', df_swap_eonia)

In [4]:
### spread zero
df_spread_zero_rate_sofr = pd.DataFrame([['1Y', 0.]], columns=['axis1', 'values']).set_index('axis1')
data_szr_sofr = qfCreateData1D('SPREAD ZERO RATE', 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD', df_spread_zero_rate_sofr)
df_spread_zero_rate_eonia = pd.DataFrame([['1Y', 0.]], columns=['axis1', 'values']).set_index('axis1')
data_szr_eonia = qfCreateData1D('SPREAD ZERO RATE', 'EONIA-1B-FLAT-OVER-EONIA-1B-ZERO-SPREAD', df_spread_zero_rate_eonia)

In [5]:
### fx rate
df_eur_usd_fx = pd.DataFrame([['0D', 1.15]], columns=['axis1', 'values']).set_index('axis1')
data_eur_usd_fx = qfCreateData1D('FX Spot Rate', 'EUR-USD', df_eur_usd_fx)

In [6]:
### funding parameter table
df_dg_usd = pd.DataFrame([['Overnight Index Swap', 'USD-SOFR-OIS', 'SOFR-1B-FLAT']], columns=['DATA TYPE', 'DATA CONVENTION', 'FUNDING IDENTIFIER'])
data_fpt_usd = qfCreateDataGeneric('DATA GENERIC', 'USD-FUNDING-PARAMETERS', df_dg_usd)
df_dg_eur = pd.DataFrame([
    ['Overnight Index Swap', 'EUR-OIS', 'EONIA-1B-FLAT'],
    ['FX Spot Rate', 'EUR-USD', 'SOFR-1B-FLAT']], columns=['DATA TYPE', 'DATA CONVENTION', 'FUNDING IDENTIFIER'])
data_fpt_eur = qfCreateDataGeneric('DATA GENERIC', 'EUR-FUNDING-PARAMETERS', df_dg_eur)

In [7]:
### pack up all data into data collection
data_collection = qfCreateDataCollection([data_swap_sofr, data_swap_eonia, data_szr_sofr, data_szr_eonia, data_fpt_usd, data_fpt_eur, data_eur_usd_fx])

### Create Model Yield Curve

In [8]:
value_date = '2026-02-11'
yc_model = qfCreateModel(value_date, 'YIELD_CURVE', data_collection, build_method_collection)
# path = 'serialized/yc_model_multi_ccys_calibrated.pickle'
# qfWriteModelObjectToFile(yc_model, path)